In [1]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

from sentence_transformers import SentenceTransformer
#导入停用词
# stop_word = [line.strip() for line in open('baidu_stopwords','r', encoding='utf-8')]
stop_word = []
PATH = "../1爬取关键词微博/temp.txt"
data_set = []  #用于存储文档
for line in open(PATH, encoding='utf-8'):
    data_set.append(line.strip())

#加载可以对句子进行embedding的模型，这里选的是SentenceTransformer中的多语言模型，因为这个转向量特别快，
#当然也可以用一些古汉语相关的模型，'SIKU-BERT/sikubert'
sentence_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
#将所有的句子都转化为embedding
embeddings = sentence_model.encode(data_set, show_progress_bar=True)
print(embeddings.shape) #输出句子的embedding的维度
print(embeddings[0]) #输出第一句话的embedding
print('句子的个数：',len(data_set)) #输出句子的个数

Batches:   0%|          | 0/4550 [00:00<?, ?it/s]

(145576, 384)
[ 1.33381829e-01 -1.02593834e-02 -3.00869256e-01 -1.53919727e-01
  4.19450691e-03  7.75943249e-02  2.67699026e-02  4.23249491e-02
  2.26036176e-01  6.20238706e-02  4.34905365e-02  4.76566032e-02
 -1.09936990e-01  1.64580077e-01 -1.40863925e-01  3.80894244e-02
 -1.07321464e-01 -1.99541688e-01 -4.65789586e-01  1.58822373e-01
 -1.94782287e-01 -1.73101097e-01  2.12811697e-02 -5.90208359e-02
 -1.24288745e-01 -3.81187513e-03 -8.36784542e-02  5.88395745e-02
 -3.14124942e-01 -7.88011064e-04 -1.68225691e-01 -1.85170978e-01
 -1.18865585e-02  2.99947709e-01 -3.34261835e-01  6.49362206e-02
 -7.16931447e-02  5.65148853e-02  2.67116576e-02 -7.27385953e-02
  1.67813629e-01 -4.14574146e-03 -1.03533268e-01  3.80443037e-02
 -8.89464021e-02  1.29088849e-01  1.36969596e-01  9.59351659e-02
 -1.59006745e-01 -3.82548243e-01 -8.86755735e-02  2.36576274e-02
  1.04460984e-01  2.83166349e-01 -1.00357592e-01  5.60864881e-02
 -1.27607137e-01 -2.93512732e-01 -8.97691865e-03  2.81805806e-02
  5.0293020

In [27]:
# from bertopic import BERTopic
# import umap 
# import hdbscan
# # Create instances of GPU-accelerated UMAP and HDBSCAN
# umap_model = umap.UMAP(n_components=5, n_neighbors=15, min_dist=0.0)
# hdbscan_model = hdbscan.HDBSCAN(min_samples=10, gen_min_span_tree=True, prediction_data=True)
# vectorizer_model = CountVectorizer(stop_words=stop_word,analyzer='word', token_pattern=u"(?u)\\b\\w+\\b")
# 
# topic_model = BERTopic(
#     # embedding_model=embedding_model,    # Step 1 - Extract embeddings
#     umap_model=umap_model,              # Step 2 - Reduce dimensionality
#     hdbscan_model=hdbscan_model,        # Step 3 - Cluster reduced embeddings
#     vectorizer_model=vectorizer_model,  # Step 4 - Tokenize topics
#     # ctfidf_model=ctfidf_model,          # Step 5 - Extract topic words
#     nr_topics=10,
#     top_n_words=30,
#     language="multilingual"
# )
# topic = topic_model.fit(data_set, embeddings)

In [2]:
#然后初始化CountVectorizer，在bertopic中计算tf-idf需要用到的，这时候要加载停用词，  作者建议在聚类之后再使用停用词，因为句子embedding需要句子完整，如果一开始就去停用词，会导致embdding语义不完整
#所以在 CountVectorizer中才使用停用词，其实也就是不计算停用词对主题类的重要性
vectorizer_model = CountVectorizer(stop_words=stop_word,analyzer='word', token_pattern=u"(?u)\\b\\w+\\b")
#这里analyzer='word', token_pattern=u"(?u)\\b\\w+\\b" 的设置就是针对汉语的，因为汉语词既可以是单字词，又可以是多字词
topic_model = BERTopic(vectorizer_model=vectorizer_model, top_n_words=30,language="multilingual")
topic = topic_model.fit(data_set, embeddings)

In [3]:
topic.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,62178,-1_就_去_没_我,"[就, 去, 没, 我, 不, 一个, 看, 说, 了, 会, 想, 得, 还是, 还, 什...","[艾滋病 想 知道 大家 都 怎么 被 感染 的, 我 知道 他 有 艾滋 但 我 还是 想..."
1,0,1522,0_他_二时_何谓_金山,"[他, 二时, 何谓, 金山, 千古, 刚刚, 发颤, 查出, 失足, 被子, 告诫, 勇于...",[他 曾 在 大 二时 嫖娼 刚刚 查出 艾滋病毒 携带 23 岁 大 三 男生 蒙 被子 ...
2,1,1508,1_同性恋_骗婚_异性恋_男,"[同性恋, 骗婚, 异性恋, 男, 金牌, 免, 高贵, 不得好死, 特权, 恶心, 出柜,...","[我 不 歧视 同性恋 也 不 歧视 艾滋 歧视 的 是 得 艾滋 的 男 同性恋, 同性恋..."
3,2,1260,2_入校_儿童_7_携带,"[入校, 儿童, 7, 携带, 无法, 岁, 孩子, 家长, 艾滋病毒, 学校, 小孩, 小...","[7 岁 艾滋病毒 携带 儿童 无法 入校 这种 父母 真的 是, 7 岁 艾滋病毒 携带 ..."
4,3,1224,3_男人_嫖娼_女人_同,"[男人, 嫖娼, 女人, 同, 男, 男性, 女, 女权, 女性, 妓女, 脏, 嫖, 男女...",[男朋友 问 我 男朋友 出轨 和 嫖娼 哪个 最 不能 接受 我 说 嫖娼 吧 你 要是 ...
...,...,...,...,...,...
1885,1884,10,1884_怜_庆_首_藏于,"[怜, 庆, 首, 藏于, 场景, 歌, 追光, 垃圾岛, 茫茫, 嗓音, 起航, 逐, 单...",[朴实无华 的 摄影 作品 不离不弃 却 触碰 到了 心底 最 柔软 的 部分 相依相偎 有...
1886,1885,10,1885_幻_仙_科技_印钞机,"[幻, 仙, 科技, 印钞机, 僭, 美联储, 基站, 工业设计, 交费, 断言, 硬件, ...",[第五代 移动 通信 技术 到底 对 人体 有 什么 健康 隐患 言之凿凿 的 人 哪个 可...
1887,1886,10,1886_求异_传播者_异性恋_顺,"[求异, 传播者, 异性恋, 顺, 主观, 色彩, 天下无双, 泣鬼神, 惊天地, 判为, ...",[而且 想说 一句话 即使 性 少数 是 艾滋病 性病 的 主要 传播者 性 少数 也 不 ...
1888,1887,10,1887_1331_用心_生日快乐_王俊,"[1331, 用心, 生日快乐, 王俊, 好棒, 恶魔, 应援, 带动, 覆盖面, 凯, 二...",[王俊 凯 王俊 凯 0921 生日快乐 好棒 呀 王俊 凯 1331 名 通过 母婴 传播...


In [5]:
first_new_topic_probs = topic_model.visualize_documents(data_set, embeddings=embeddings)
first_new_topic_probs.write_html('visualize_documents.html')

In [6]:

out = topic_model.visualize_hierarchy(top_n_topics=380)
out.write_html('visualize_hierarchy.html')

In [ ]:
first_new_topic_probs = topic_model.visualize_heatmap()
first_new_topic_probs.write_html('visualize_heatmap.html')

In [ ]:
first_new_topic_probs = topic_model.visualize_term_rank()
first_new_topic_probs.write_html('visualize_term_rank.html')

In [ ]:
first_new_topic_probs = topic_model.visualize_topics()
first_new_topic_probs.write_html('visualize_topics.html')


In [4]:

hierarchical_topics = topic_model.hierarchical_topics(data_set)


100%|██████████| 2310/2310 [00:59<00:00, 39.06it/s]


In [5]:
hierarchical_topics

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance
2309,4620,吴宣仪_网易_谢谢_狗_尤长靖,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",4405,吴宣仪_火箭_笑语_罗_小昂,4619,网易_谢谢_狗_尤长靖_邓伦,3.270074
2308,4619,网易_谢谢_狗_尤长靖_邓伦,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",4618,网易_谢谢_狗_邓伦_大熊猫,4616,心瘾_尤长靖_韬_黄子韬_演唱会,3.069573
2307,4618,网易_谢谢_狗_邓伦_大熊猫,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",4617,网易_谢谢_狗_邓伦_大熊猫,4520,龚俊_子俊俊_龚俊西_蒙俊_韩烨,3.062487
2306,4617,网易_谢谢_狗_邓伦_大熊猫,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",4615,网易_谢谢_狗_大熊猫_邓伦,2745,热巴_迪丽_力士_金纺迪丽_李长歌,2.809955
2305,4616,心瘾_尤长靖_韬_黄子韬_演唱会,"[91, 165, 166, 182, 189, 197, 245, 301, 318, 3...",4613,尤长靖_西柚_新歌_亚洲_榜,4610,心瘾_韬_黄子韬_冰激凌_演唱会,2.759829
...,...,...,...,...,...,...,...,...
4,2315,心瘾_韬_韬黄子_韬刷_韬才,"[590, 1384, 2139]",2311,心瘾_韬_韬黄子_韬刷_胡汉三,1384,心瘾_韬_韬才_胡汉三_下雨天,0.053366
3,2314,缺土_金旺_属猪_双子_牧民,"[646, 1872]",646,缺土_金旺_属猪_双子_牧民,1872,缺土_金旺_属猪_双子_响爷,0.050236
2,2313,脖_鸭哥_精武_精研_五香,"[1097, 1366]",1366,脖_鸭哥_精武_精研_五香,1097,脖_鸭哥_精武_精研_五香,0.030981
1,2312,阳阳_阳光_范_周翊然_霸道,"[261, 1700]",261,阳阳_阳光_范_周翊然_霸道,1700,阳阳_晚安_阳光_宝贝_时间,0.029309


In [7]:
import  jiagu
sss = jiagu.sentiment('只字不提艾滋阳性好吧为了稳定')


In [11]:
sss[1]

0.6527195473956697